# 06 — Unifactorial HRV analysis | Extended EDA

**Scope:** each of the seven HRV outcomes is analyzed **separately** against individual demographic, blood-pressure, and BMI factors. No joint HRV analysis, factorial interactions, or participant reclassification is performed.

**Input:** `../../outputs/dataset_final.csv` (Notebook 03).  
**Output directory:** `../../outputs/06_unifactorial_hrv_analysis/`.

**Categorical analysis (retained from v1):** descriptive statistics, group sample sizes, distribution comparisons (histograms, ECDFs, boxplots and violins), Kruskal–Wallis, rank-based epsilon-squared, Brown–Forsythe, tie-corrected Dunn post hoc (Mann–Whitney for sex), Cliff's delta, Kolmogorov–Smirnov, Wasserstein, and ordered-factor trends.

**Additions:** stratified bootstrap 95% confidence intervals for Kruskal epsilon-squared, Cliff's delta and median differences; continuous-factor Spearman correlations and bootstrap intervals; scatterplots with robust LOWESS trends and binned median/IQR summaries; sensitivity to categories with n < 10 and to trimming the lowest/highest 1% of each HRV outcome; and cross-factor association summary heatmaps/tables.

**Interpretation:** entirely exploratory and cross-sectional. Relative group contrasts are not clinical HRV thresholds. Four-variable FDR families are reported explicitly; small categories are retained in the primary analyses and flagged. A statistically significant global test does not identify which groups differ. Clinical and causal interpretations require independent evidence.

**Reproducibility:** bootstrap random seed and number of resamples are fixed below. Uncertainty intervals are nonparametric percentile bootstrap intervals, not bias-corrected intervals. No source observations are modified or dropped from the primary dataset.

In [1]:
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from statsmodels.nonparametric.smoothers_lowess import lowess
from scipy import stats
from statsmodels.stats.multitest import multipletests
from IPython.display import display

INPUT_FILE = Path("../../outputs/dataset_final.csv")
OUTPUT_DIR = Path("../../eda_results/06_unifactorial_hrv_analysis")
FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
GLOBAL_FIGURE_DIR = FIGURE_DIR / "_global"
GLOBAL_FIGURE_DIR.mkdir(exist_ok=True)
FACTOR_OVERVIEW_DIR = GLOBAL_FIGURE_DIR / "factor_overviews"
FACTOR_OVERVIEW_DIR.mkdir(exist_ok=True)
SYNTHESIS_FIGURE_DIR = GLOBAL_FIGURE_DIR / "synthesis"
SYNTHESIS_FIGURE_DIR.mkdir(exist_ok=True)

RANDOM_SEED = 20261125
BOOTSTRAP_REPLICATES = 1000
OMNIBUS_BOOTSTRAP_REPLICATES = 600
MIN_SENSITIVITY_GROUP_N = 10

HRV = {
    "t2m_pre_mean_rr": ("Mean RR", "ms"),
    "t2m_pre_mean_hr": ("Mean HR", "bpm"),
    "t2m_pre_sdnn": ("SDNN", "ms"),
    "t2m_pre_rmssd": ("RMSSD", "ms"),
    "t2m_pre_hf": ("HF", "ms²"),
    "t2m_pre_lf": ("LF", "ms²"),
    "t2m_pre_vlf": ("VLF", "ms²"),
}
FACTORS = {
    "age_group": ["60-69", "70-79", "80+"],
    "sex_category": ["Female", "Male"],
    "bp_category": ["Optimal", "Normal", "High-normal", "Stage 1", "Stage 2", "Stage 3"],
    "bmi_category": ["Underweight", "Normal", "Overweight", "Obesity"],
}
CONTINUOUS_FACTORS = {
    "age": ("Age", "years", "age_group"),
    "imc": ("BMI", "kg/m²", "bmi_category"),
    "bp_systolic": ("Systolic blood pressure", "mmHg", "bp_category"),
    "bp_diastolic": ("Diastolic blood pressure", "mmHg", "bp_category"),
}
FACTOR_NAMES = {
    "age_group": "Age group",
    "sex_category": "Sex",
    "bp_category": "Blood pressure category",
    "bmi_category": "BMI category",
}

sns.set_theme(style="whitegrid", context="notebook", font_scale=1.05)
plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "savefig.facecolor": "white", "axes.titleweight": "semibold",
    "axes.spines.top": False, "axes.spines.right": False,
    "grid.alpha": 0.20, "savefig.dpi": 180,
})

PALETTES = {
    "age_group": sns.color_palette("viridis", n_colors=3),
    "sex_category": sns.color_palette("colorblind", n_colors=2),
    "bp_category": sns.color_palette("crest", n_colors=6),
    "bmi_category": sns.color_palette("flare", n_colors=4),
}
COLORS = {
    factor: dict(zip(levels, PALETTES[factor]))
    for factor, levels in FACTORS.items()
}

def slugify(text):
    return text.lower().replace(" ", "_").replace("/", "_")

HRV_FIGURE_DIRS = {}
for col, (label, _unit) in HRV.items():
    base = FIGURE_DIR / slugify(label)
    (base / "categorical").mkdir(parents=True, exist_ok=True)
    (base / "continuous").mkdir(parents=True, exist_ok=True)
    HRV_FIGURE_DIRS[col] = {
        "base": base,
        "categorical": base / "categorical",
        "continuous": base / "continuous",
    }

df = pd.read_csv(INPUT_FILE)
required = list(HRV) + list(FACTORS) + list(CONTINUOUS_FACTORS)
missing = [col for col in required if col not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")
for col in list(HRV) + list(CONTINUOUS_FACTORS):
    if not pd.api.types.is_numeric_dtype(df[col]):
        raise TypeError(f"Expected numeric data: {col}")
for factor, levels in FACTORS.items():
    unexpected = set(df[factor].dropna().unique()) - set(levels)
    if unexpected:
        raise ValueError(f"Unexpected categories in {factor}: {sorted(unexpected)}")

print(f"Participants: {len(df)} | HRV variables: {len(HRV)} | Factors: {len(FACTORS)}")
print(f"Output folder: {OUTPUT_DIR}")
print("Figure folders are organized by HRV variable under outputs/06_unifactorial_hrv_analysis/figures/")

Participants: 476 | HRV variables: 7 | Factors: 4
Output folder: ../../eda_results/06_unifactorial_hrv_analysis
Figure folders are organized by HRV variable under outputs/06_unifactorial_hrv_analysis/figures/


## 1. Factor sizes and group-wise HRV distributions

Descriptive statistics are calculated for **each HRV measurement within each factor level**. Small groups are explicitly identified. No values are removed, winsorized, or normalized.

In [2]:
counts = []
for factor, levels in FACTORS.items():
    for level in levels:
        n = int((df[factor] == level).sum())
        counts.append({
            "factor": factor, "level": level, "n": n,
            "percent": 100 * n / len(df), "small_n_below_5": n < 5,
        })
count_table = pd.DataFrame(counts)
count_table.to_csv(OUTPUT_DIR / "factor_counts.csv", index=False)
display(count_table)

rows = []
for factor, levels in FACTORS.items():
    for col, (label, unit) in HRV.items():
        for level in levels:
            v = df.loc[df[factor] == level, col].dropna()
            q1, q3 = v.quantile([0.25, 0.75]) if len(v) else (np.nan, np.nan)
            rows.append({
                "factor": factor, "level": level, "hrv_variable": col,
                "hrv_label": label, "unit": unit,
                "n": len(v), "missing_n": int(((df[factor] == level) & df[col].isna()).sum()),
                "mean": v.mean(), "median": v.median(), "std": v.std(),
                "q25": q1, "q75": q3, "iqr": q3 - q1,
                "p05": v.quantile(0.05), "p95": v.quantile(0.95),
                "min": v.min(), "max": v.max(),
                "skewness": v.skew() if len(v) >= 3 else np.nan,
                "small_n_below_5": len(v) < 5,
            })
group_summary = pd.DataFrame(rows)
group_summary.to_csv(OUTPUT_DIR / "hrv_group_descriptives.csv", index=False)
print(f"Group-level descriptive rows: {len(group_summary)}")
display(group_summary.head(12).round(3))

,factor,level,n,percent,small_n_below_5
0,age_group,60-69,198,41.596639,False
1,age_group,70-79,241,50.630252,False
2,age_group,80+,37,7.773109,False
3,sex_category,Female,333,69.957983,False
4,sex_category,Male,143,30.042017,False
5,bp_category,Optimal,98,20.588235,False
6,bp_category,Normal,123,25.840336,False
7,bp_category,High-normal,106,22.268908,False
8,bp_category,Stage 1,111,23.319328,False
9,bp_category,Stage 2,34,7.142857,False


Group-level descriptive rows: 105


,factor,level,hrv_variable,hrv_label,unit,n,missing_n,mean,median,std,q25,q75,iqr,p05,p95,min,max,skewness,small_n_below_5
0,age_group,60-69,t2m_pre_mean_rr,Mean RR,ms,198,0,843.126,833.50,115.133,763.250,923.00,159.750,668.80,1036.000,577.0,1296.0,0.313,False
1,age_group,70-79,t2m_pre_mean_rr,Mean RR,ms,241,0,870.776,875.00,131.707,784.000,961.00,177.000,652.00,1080.000,468.0,1203.0,-0.173,False
2,age_group,80+,t2m_pre_mean_rr,Mean RR,ms,37,0,882.595,888.00,115.033,785.000,987.00,202.000,685.00,1047.000,664.0,1078.0,-0.171,False
3,age_group,60-69,t2m_pre_mean_hr,Mean HR,bpm,198,0,72.545,72.00,10.081,65.000,79.00,14.000,58.00,90.150,46.0,104.0,0.523,False
4,age_group,70-79,t2m_pre_mean_hr,Mean HR,bpm,241,0,70.635,69.00,11.912,62.000,77.00,15.000,56.00,92.000,50.0,128.0,1.333,False
5,age_group,80+,t2m_pre_mean_hr,Mean HR,bpm,37,0,69.162,68.00,9.406,61.000,76.00,15.000,57.00,87.200,56.0,90.0,0.569,False
6,age_group,60-69,t2m_pre_sdnn,SDNN,ms,198,0,141.613,149.00,84.257,82.500,203.75,121.250,12.61,270.150,4.8,430.0,0.121,False
7,age_group,70-79,t2m_pre_sdnn,SDNN,ms,241,0,136.656,139.00,100.987,61.000,191.00,130.000,9.90,306.000,6.5,626.0,1.191,False
8,age_group,80+,t2m_pre_sdnn,SDNN,ms,37,0,111.511,97.00,88.251,14.500,156.00,141.500,8.26,262.800,6.4,334.0,0.721,False
9,age_group,60-69,t2m_pre_rmssd,RMSSD,ms,198,0,14.557,14.05,6.830,10.225,18.20,7.975,4.61,26.965,1.0,38.2,0.635,False


## 2. Omnibus location and dispersion comparisons

**Kruskal–Wallis:** nonparametric comparison across all observed levels of each factor. **Epsilon-squared:** exploratory magnitude of the global rank-based difference. **Brown–Forsythe (median-centered Levene):** evidence of unequal dispersion among groups. Kruskal–Wallis tests differences in rank distributions, and does not by itself prove a pure median shift when distributions have different shapes.

The `smallest_group_n` and `small_group_caution` fields make sparse-category comparisons visible. P-values are exploratory, especially when a factor contains a four-participant category.

In [3]:
omnibus_rows = []
for factor, levels in FACTORS.items():
    for col, (label, unit) in HRV.items():
        d = df[[factor, col]].dropna()
        groups = [d.loc[d[factor] == level, col].to_numpy() for level in levels]
        groups = [x for x in groups if len(x)]
        n_total, k = sum(len(x) for x in groups), len(groups)
        h, kw_p = stats.kruskal(*groups)
        lev_stat, lev_p = stats.levene(*groups, center="median")
        epsilon2 = max(0.0, (h - k + 1) / (n_total - k))
        omnibus_rows.append({
            "factor": factor, "hrv_variable": col, "hrv_label": label,
            "n": n_total, "groups": k, "smallest_group_n": min(map(len, groups)),
            "small_group_caution": min(map(len, groups)) < 5,
            "kruskal_h": h, "kruskal_p": kw_p,
            "epsilon_squared": epsilon2,
            "brown_forsythe_statistic": lev_stat,
            "brown_forsythe_p": lev_p,
        })
omnibus = pd.DataFrame(omnibus_rows)
omnibus["kruskal_q_bh_28_tests"] = multipletests(omnibus["kruskal_p"], method="fdr_bh")[1]
omnibus["brown_forsythe_q_bh_28_tests"] = multipletests(omnibus["brown_forsythe_p"], method="fdr_bh")[1]
omnibus.to_csv(OUTPUT_DIR / "omnibus_tests.csv", index=False)
display(omnibus.round(4))

,factor,hrv_variable,hrv_label,n,groups,smallest_group_n,small_group_caution,kruskal_h,kruskal_p,epsilon_squared,brown_forsythe_statistic,brown_forsythe_p,kruskal_q_bh_28_tests,brown_forsythe_q_bh_28_tests
0,age_group,t2m_pre_mean_rr,Mean RR,476,3,37,False,8.1405,0.0171,0.0130,1.7345,0.1776,0.0598,0.5800
1,age_group,t2m_pre_mean_hr,Mean HR,476,3,37,False,8.3249,0.0156,0.0134,1.0254,0.3595,0.0598,0.7189
2,age_group,t2m_pre_sdnn,SDNN,476,3,37,False,6.5063,0.0387,0.0095,1.5526,0.2128,0.0984,0.5800
3,age_group,t2m_pre_rmssd,RMSSD,476,3,37,False,3.9959,0.1356,0.0042,0.6370,0.5293,0.2420,0.7411
4,age_group,t2m_pre_hf,HF,476,3,37,False,12.2757,0.0022,0.0217,0.9030,0.4060,0.0151,0.7259
5,age_group,t2m_pre_lf,LF,476,3,37,False,13.7010,0.0011,0.0247,0.8816,0.4148,0.0099,0.7259
6,age_group,t2m_pre_vlf,VLF,476,3,37,False,7.0610,0.0293,0.0107,1.7836,0.1692,0.0820,0.5800
7,sex_category,t2m_pre_mean_rr,Mean RR,476,2,143,False,0.0438,0.8342,0.0000,2.1420,0.1440,0.8984,0.5800
8,sex_category,t2m_pre_mean_hr,Mean HR,476,2,143,False,0.0685,0.7935,0.0000,1.4421,0.2304,0.8887,0.5800
9,sex_category,t2m_pre_sdnn,SDNN,476,2,143,False,42.2677,0.0000,0.0871,4.5134,0.0341,0.0000,0.5800


### 2.1 Bootstrap uncertainty for global effects

A stratified percentile bootstrap (resampling independently **within** each factor level) estimates a 95% interval for the rank-based Kruskal epsilon-squared. Values are nonnegative by construction. Extremely small groups yield unstable resampling intervals, so their uncertainty must be interpreted cautiously.

In [4]:
def percentile_interval(values):
    values = np.asarray(values, dtype=float)
    finite = values[np.isfinite(values)]
    if len(finite) == 0:
        return np.nan, np.nan
    return tuple(np.quantile(finite, [0.025, 0.975]))

kw_bootstrap = []
for j, row in omnibus.iterrows():
    groups = [df.loc[df[row["factor"]] == level, row["hrv_variable"]].dropna().to_numpy()
              for level in FACTORS[row["factor"]]]
    groups = [g for g in groups if len(g)]
    n = sum(map(len, groups))
    k = len(groups)
    rng = np.random.default_rng(RANDOM_SEED + 10000 + j)
    samples = np.empty(OMNIBUS_BOOTSTRAP_REPLICATES)
    for b in range(OMNIBUS_BOOTSTRAP_REPLICATES):
        resampled = [rng.choice(g, size=len(g), replace=True) for g in groups]
        h = stats.kruskal(*resampled).statistic
        samples[b] = max(0.0, (h - k + 1) / (n - k))
    kw_bootstrap.append(percentile_interval(samples))

omnibus[["epsilon_squared_ci95_lower", "epsilon_squared_ci95_upper"]] = np.asarray(kw_bootstrap)
omnibus.to_csv(OUTPUT_DIR / "omnibus_tests.csv", index=False)
print(f"Added {OMNIBUS_BOOTSTRAP_REPLICATES} stratified bootstrap replicates for each of {len(omnibus)} omnibus effects.")

Added 600 stratified bootstrap replicates for each of 28 omnibus effects.


## 3. Pairwise comparisons and effect sizes

- **Dunn's test:** post hoc comparison of average pooled ranks, with tie correction, for factors with three or more levels.
- **Mann–Whitney U:** two-sided comparison for the two-level sex factor.
- **Cliff's delta:** signed pairwise effect size, positive when the first level tends to have larger values. The sign depends on level order, not on an outcome's clinical interpretation.
- **Two-sample KS:** comparison of empirical distributions, with the KS statistic and p-value. **Wasserstein distance:** magnitude of distributional separation in the original measurement units.

Dunn/Mann–Whitney and KS p-values are each FDR-adjusted **within one HRV outcome and one factor**. A small group is retained in the calculation but flagged; estimates involving such groups should be interpreted especially cautiously.

In [5]:
# Tie-corrected pairwise Dunn comparisons using the pooled ranks.
def dunn_pvalues(values, labels, levels):
    ranks = stats.rankdata(values)
    n = len(values)
    counts = np.array([(labels == level).sum() for level in levels])
    mean_ranks = np.array([ranks[labels == level].mean() for level in levels])
    tie_counts = np.unique(values, return_counts=True)[1]
    tie_term = (tie_counts**3 - tie_counts).sum() / (12 * (n - 1))
    rank_variance = n * (n + 1) / 12 - tie_term
    pvalues = {}
    for i, j in combinations(range(len(levels)), 2):
        se = np.sqrt(rank_variance * (1 / counts[i] + 1 / counts[j]))
        z = (mean_ranks[i] - mean_ranks[j]) / se
        pvalues[(levels[i], levels[j])] = 2 * stats.norm.sf(abs(z))
    return pvalues

pairwise_rows = []
for factor, levels in FACTORS.items():
    for col, (label, unit) in HRV.items():
        d = df[[factor, col]].dropna()
        observed = [level for level in levels if (d[factor] == level).any()]
        if len(observed) >= 3:
            all_values = d[col].to_numpy()
            all_labels = d[factor].to_numpy()
            dunn = dunn_pvalues(all_values, all_labels, observed)
        for left, right in combinations(observed, 2):
            a = d.loc[d[factor] == left, col].to_numpy()
            b = d.loc[d[factor] == right, col].to_numpy()
            mw = stats.mannwhitneyu(a, b, alternative="two-sided", method="asymptotic")
            cliffs_delta = 2 * mw.statistic / (len(a) * len(b)) - 1
            ks = stats.ks_2samp(a, b, alternative="two-sided", method="auto")
            pairwise_rows.append({
                "factor": factor, "hrv_variable": col, "unit": unit,
                "level_1": left, "level_2": right,
                "n_1": len(a), "n_2": len(b),
                "small_group_caution": min(len(a), len(b)) < 5,
                "median_1": np.median(a), "median_2": np.median(b),
                "median_difference_1_minus_2": np.median(a) - np.median(b),
                "comparison_test": "Dunn" if len(observed) >= 3 else "Mann-Whitney U",
                "comparison_p": dunn[(left, right)] if len(observed) >= 3 else mw.pvalue,
                "cliffs_delta_1_vs_2": cliffs_delta,
                "ks_statistic": ks.statistic, "ks_p": ks.pvalue,
                "wasserstein_distance": stats.wasserstein_distance(a, b),
            })
pairwise = pd.DataFrame(pairwise_rows)
pairwise["comparison_q_bh_within_family"] = np.nan
pairwise["ks_q_bh_within_family"] = np.nan
for (factor, hrv), block in pairwise.groupby(["factor", "hrv_variable"], sort=False):
    idx = block.index
    pairwise.loc[idx, "comparison_q_bh_within_family"] = multipletests(block["comparison_p"], method="fdr_bh")[1]
    pairwise.loc[idx, "ks_q_bh_within_family"] = multipletests(block["ks_p"], method="fdr_bh")[1]
pairwise.to_csv(OUTPUT_DIR / "pairwise_comparisons.csv", index=False)
print(f"Pairwise comparisons: {len(pairwise)}")
display(pairwise.head(14).round(4))

Pairwise comparisons: 175


,factor,hrv_variable,unit,level_1,level_2,n_1,n_2,small_group_caution,median_1,median_2,median_difference_1_minus_2,comparison_test,comparison_p,cliffs_delta_1_vs_2,ks_statistic,ks_p,wasserstein_distance,comparison_q_bh_within_family,ks_q_bh_within_family
0,age_group,t2m_pre_mean_rr,ms,60-69,70-79,198,241,False,833.50,875.0,-41.50,Dunn,0.0101,-0.1417,0.1383,0.0277,33.4724,0.0303,0.0653
1,age_group,t2m_pre_mean_rr,ms,60-69,80+,198,37,False,833.50,888.0,-54.50,Dunn,0.0528,-0.2058,0.2413,0.0435,42.3504,0.0792,0.0653
2,age_group,t2m_pre_mean_rr,ms,70-79,80+,241,37,False,875.00,888.0,-13.00,Dunn,0.5704,-0.0535,0.1154,0.7382,22.5371,0.5704,0.7382
3,age_group,t2m_pre_mean_hr,bpm,60-69,70-79,198,241,False,72.00,69.0,3.00,Dunn,0.0089,0.1440,0.1300,0.0454,2.7717,0.0268,0.1323
4,age_group,t2m_pre_mean_hr,bpm,60-69,80+,198,37,False,72.00,68.0,4.00,Dunn,0.0531,0.2058,0.2176,0.0882,3.5146,0.0797,0.1323
5,age_group,t2m_pre_mean_hr,bpm,70-79,80+,241,37,False,69.00,68.0,1.00,Dunn,0.5884,0.0505,0.0988,0.8783,2.0525,0.5884,0.8783
6,age_group,t2m_pre_sdnn,ms,60-69,70-79,198,241,False,149.00,139.0,10.00,Dunn,0.1922,0.0727,0.0901,0.3151,16.7314,0.1922,0.3151
7,age_group,t2m_pre_sdnn,ms,60-69,80+,198,37,False,149.00,97.0,52.00,Dunn,0.0132,0.2539,0.2630,0.0214,33.0330,0.0396,0.0642
8,age_group,t2m_pre_sdnn,ms,70-79,80+,241,37,False,139.00,97.0,42.00,Dunn,0.0709,0.1865,0.1859,0.1896,25.3859,0.1064,0.2845
9,age_group,t2m_pre_rmssd,ms,60-69,70-79,198,241,False,14.05,13.6,0.45,Dunn,0.7520,0.0170,0.0526,0.9035,0.9787,0.7520,0.9035


### 3.1 Confidence intervals for pairwise effect sizes

**Cliff's delta** is computed as the probability of first-group superiority minus the probability of second-group superiority, with ties contributing zero. The table includes 95% percentile bootstrap intervals for Cliff's delta and the difference in group medians (first minus second). Pairwise p-values are also FDR-adjusted across **all 175 comparisons**, in addition to the original within-HRV-by-factor adjustment. These are distinct correction scopes.

In [6]:
def cliffs_delta_fast(a, b):
    # Count wins/ties by binary search; equal observations count as ties.
    b_sorted = np.sort(b)
    below = np.searchsorted(b_sorted, a, side="left")
    equal = np.searchsorted(b_sorted, a, side="right") - below
    wins = below.sum()
    ties = equal.sum()
    return (2 * wins + ties) / (len(a) * len(b)) - 1

cliff_ci, median_ci = [], []
for j, row in pairwise.iterrows():
    a = df.loc[df[row["factor"]] == row["level_1"], row["hrv_variable"]].dropna().to_numpy()
    b = df.loc[df[row["factor"]] == row["level_2"], row["hrv_variable"]].dropna().to_numpy()
    rng = np.random.default_rng(RANDOM_SEED + 30000 + j)
    cliff_samples = np.empty(BOOTSTRAP_REPLICATES)
    median_samples = np.empty(BOOTSTRAP_REPLICATES)
    for i in range(BOOTSTRAP_REPLICATES):
        aa = rng.choice(a, size=len(a), replace=True)
        bb = rng.choice(b, size=len(b), replace=True)
        cliff_samples[i] = cliffs_delta_fast(aa, bb)
        median_samples[i] = np.median(aa) - np.median(bb)
    cliff_ci.append(percentile_interval(cliff_samples))
    median_ci.append(percentile_interval(median_samples))

pairwise[["cliffs_delta_ci95_lower", "cliffs_delta_ci95_upper"]] = np.asarray(cliff_ci)
pairwise[["median_difference_ci95_lower", "median_difference_ci95_upper"]] = np.asarray(median_ci)
pairwise["comparison_q_bh_all_175_tests"] = multipletests(pairwise["comparison_p"], method="fdr_bh")[1]
pairwise["ks_q_bh_all_175_tests"] = multipletests(pairwise["ks_p"], method="fdr_bh")[1]
pairwise["bootstrap_replicates"] = BOOTSTRAP_REPLICATES
pairwise.to_csv(OUTPUT_DIR / "pairwise_comparisons.csv", index=False)
print(f"Computed Cliff's delta and median-difference confidence intervals for {len(pairwise)} pairwise comparisons.")
display(pairwise[["factor", "hrv_variable", "level_1", "level_2", "n_1", "n_2", "cliffs_delta_1_vs_2", "cliffs_delta_ci95_lower", "cliffs_delta_ci95_upper", "comparison_q_bh_all_175_tests"]].head(12).round(3))

Computed Cliff's delta and median-difference confidence intervals for 175 pairwise comparisons.


,factor,hrv_variable,level_1,level_2,n_1,n_2,cliffs_delta_1_vs_2,cliffs_delta_ci95_lower,cliffs_delta_ci95_upper,comparison_q_bh_all_175_tests
0,age_group,t2m_pre_mean_rr,60-69,70-79,198,241,-0.142,-0.248,-0.042,0.173
1,age_group,t2m_pre_mean_rr,60-69,80+,198,37,-0.206,-0.399,0.003,0.314
2,age_group,t2m_pre_mean_rr,70-79,80+,241,37,-0.053,-0.232,0.147,0.792
3,age_group,t2m_pre_mean_hr,60-69,70-79,198,241,0.144,0.041,0.252,0.173
4,age_group,t2m_pre_mean_hr,60-69,80+,198,37,0.206,-0.005,0.395,0.314
5,age_group,t2m_pre_mean_hr,70-79,80+,241,37,0.050,-0.154,0.250,0.798
6,age_group,t2m_pre_sdnn,60-69,70-79,198,241,0.073,-0.034,0.179,0.533
7,age_group,t2m_pre_sdnn,60-69,80+,198,37,0.254,0.049,0.457,0.173
8,age_group,t2m_pre_sdnn,70-79,80+,241,37,0.186,-0.011,0.389,0.316
9,age_group,t2m_pre_rmssd,60-69,70-79,198,241,0.017,-0.090,0.130,0.895


## 4. Ordered-factor trends

For age group, blood-pressure category, and BMI category, Spearman's rho quantifies **monotonic** associations between ordered categories and individual HRV measurements. It does not imply linearity or causality. Sex is not treated as an ordinal factor. P-values are FDR-adjusted across these 21 exploratory tests.

In [7]:
trend_rows = []
for factor in ("age_group", "bp_category", "bmi_category"):
    scores = df[factor].map({level: idx for idx, level in enumerate(FACTORS[factor])})
    for col in HRV:
        valid = scores.notna() & df[col].notna()
        rho, p = stats.spearmanr(scores[valid], df.loc[valid, col])
        trend_rows.append({
            "factor": factor, "hrv_variable": col,
            "n": int(valid.sum()), "spearman_rho": rho, "p": p,
            "direction": "positive" if rho > 0 else "negative" if rho < 0 else "none",
        })
trends = pd.DataFrame(trend_rows)
trends["q_bh_21_tests"] = multipletests(trends["p"], method="fdr_bh")[1]
trends.to_csv(OUTPUT_DIR / "ordered_factor_trends.csv", index=False)
display(trends.round(4))

,factor,hrv_variable,n,spearman_rho,p,direction,q_bh_21_tests
0,age_group,t2m_pre_mean_rr,476,0.1303,0.0044,positive,0.0232
1,age_group,t2m_pre_mean_hr,476,-0.1316,0.0040,negative,0.0232
2,age_group,t2m_pre_sdnn,476,-0.1034,0.0241,negative,0.0844
3,age_group,t2m_pre_rmssd,476,-0.0594,0.1957,negative,0.4565
4,age_group,t2m_pre_hf,476,-0.1556,0.0007,negative,0.0069
5,age_group,t2m_pre_lf,476,-0.1599,0.0005,negative,0.0069
6,age_group,t2m_pre_vlf,476,-0.1131,0.0136,negative,0.0571
7,bp_category,t2m_pre_mean_rr,476,-0.0736,0.1088,negative,0.2855
8,bp_category,t2m_pre_mean_hr,476,0.0749,0.1028,positive,0.2855
9,bp_category,t2m_pre_sdnn,476,-0.0510,0.2669,negative,0.4943


## 5. Continuous-factor analyses

Age, BMI, systolic blood pressure, and diastolic blood pressure are also analyzed in their **original continuous form**. Each HRV outcome is assessed independently using Spearman's rho with a 95% case-resampling bootstrap confidence interval. A separate BH-FDR family covers the 28 continuous associations. These analyses complement categorical contrasts and avoid relying exclusively on category boundaries. Sex remains categorical. Neither adjusted regression nor multifactorial interaction tests are performed here.

In [8]:
continuous_rows = []
for j, (x, (x_label, x_unit, color_factor)) in enumerate(CONTINUOUS_FACTORS.items()):
    for k, (y, (y_label, y_unit)) in enumerate(HRV.items()):
        d = df[[x, y]].dropna()
        xx, yy = d[x].to_numpy(), d[y].to_numpy()
        rho, p = stats.spearmanr(xx, yy)
        rng = np.random.default_rng(RANDOM_SEED + 50000 + j * len(HRV) + k)
        bootstrap = np.empty(BOOTSTRAP_REPLICATES)
        for i in range(BOOTSTRAP_REPLICATES):
            indices = rng.integers(0, len(d), size=len(d))
            bootstrap[i] = stats.spearmanr(xx[indices], yy[indices]).statistic
        ci_low, ci_high = percentile_interval(bootstrap)
        continuous_rows.append({
            "factor_continuous": x, "factor_label": x_label, "factor_unit": x_unit,
            "related_category_factor": color_factor, "hrv_variable": y,
            "hrv_label": y_label, "hrv_unit": y_unit, "n": len(d),
            "spearman_rho": rho, "spearman_ci95_lower": ci_low,
            "spearman_ci95_upper": ci_high, "spearman_p": p,
            "bootstrap_replicates": BOOTSTRAP_REPLICATES,
        })
continuous = pd.DataFrame(continuous_rows)
continuous["spearman_q_bh_28_tests"] = multipletests(continuous["spearman_p"], method="fdr_bh")[1]
continuous.to_csv(OUTPUT_DIR / "continuous_factor_associations.csv", index=False)
print(f"Continuous-factor tests: {len(continuous)}")
display(continuous.round(4))

Continuous-factor tests: 28


,factor_continuous,factor_label,factor_unit,related_category_factor,hrv_variable,hrv_label,hrv_unit,n,spearman_rho,spearman_ci95_lower,spearman_ci95_upper,spearman_p,bootstrap_replicates,spearman_q_bh_28_tests
0,age,Age,years,age_group,t2m_pre_mean_rr,Mean RR,ms,476,0.1335,0.0517,0.2229,0.0035,1000,0.0246
1,age,Age,years,age_group,t2m_pre_mean_hr,Mean HR,bpm,476,-0.1340,-0.2156,-0.0499,0.0034,1000,0.0246
2,age,Age,years,age_group,t2m_pre_sdnn,SDNN,ms,476,-0.0466,-0.1442,0.0439,0.3107,1000,0.7411
3,age,Age,years,age_group,t2m_pre_rmssd,RMSSD,ms,476,-0.0267,-0.1139,0.0651,0.5609,1000,0.7417
4,age,Age,years,age_group,t2m_pre_hf,HF,ms²,476,-0.1232,-0.2099,-0.0366,0.0071,1000,0.0398
5,age,Age,years,age_group,t2m_pre_lf,LF,ms²,476,-0.1031,-0.1974,-0.0132,0.0245,1000,0.1144
6,age,Age,years,age_group,t2m_pre_vlf,VLF,ms²,476,-0.0758,-0.1623,0.0087,0.0987,1000,0.3919
7,imc,BMI,kg/m²,bmi_category,t2m_pre_mean_rr,Mean RR,ms,476,-0.0393,-0.1293,0.0489,0.3926,1000,0.7411
8,imc,BMI,kg/m²,bmi_category,t2m_pre_mean_hr,Mean HR,bpm,476,0.0372,-0.0523,0.1253,0.4178,1000,0.7411
9,imc,BMI,kg/m²,bmi_category,t2m_pre_sdnn,SDNN,ms,476,-0.0087,-0.0953,0.0841,0.8499,1000,0.8499


## 6. Detailed distribution figures

For every one of the **28 factor-by-HRV analyses**, a four-panel figure compares:

1. Overlaid frequency-normalized histograms (one density per factor level).
2. Empirical cumulative distribution functions (ECDFs).
3. Boxplots with individual observations.
4. Violin plots, shown as complementary descriptions of distribution shape.

Axes are switched to log scale **for display only** when an HRV variable spans a large positive range; the CSV data and statistical tests always use the original values. This is indicated in the relevant panel titles. The color of every category is held constant across figures for the same factor. Violin estimates for very small groups (e.g., four people in Stage 3 BP) are descriptive only.

In [9]:
def use_log_axis(column):
    v = df[column].dropna()
    return (v.min() > 0) and (v.quantile(0.99) / v.quantile(0.01) >= 30)

for factor, levels in FACTORS.items():
    colors = COLORS[factor]
    for col, (label, unit) in HRV.items():
        d = df[[factor, col]].dropna()
        log_axis = use_log_axis(col)
        outcome = omnibus.loc[(omnibus["factor"] == factor) & (omnibus["hrv_variable"] == col)].iloc[0]
        fig, axs = plt.subplots(2, 2, figsize=(17, 11.5))
        ax_hist, ax_ecdf, ax_box, ax_violin = axs.flat
        axis_label = f"{label} ({unit})"

        sns.histplot(data=d, x=col, hue=factor, hue_order=levels, palette=colors,
                     stat="density", common_norm=False, element="step", fill=False,
                     bins=28, linewidth=1.9, log_scale=(10 if log_axis else None), ax=ax_hist)
        ax_hist.set(title="Distribution comparison" + ("  |  log x-axis" if log_axis else ""),
                    xlabel=axis_label, ylabel="Density")
        if ax_hist.legend_:
            ax_hist.legend_.set_title(FACTOR_NAMES[factor])

        sns.ecdfplot(data=d, x=col, hue=factor, hue_order=levels, palette=colors,
                     linewidth=2.0, ax=ax_ecdf)
        if log_axis:
            ax_ecdf.set_xscale("log")
        ax_ecdf.set(title="Empirical cumulative distributions" + ("  |  log x-axis" if log_axis else ""),
                    xlabel=axis_label, ylabel="Cumulative proportion", ylim=(-0.02, 1.02))
        if ax_ecdf.legend_:
            ax_ecdf.legend_.set_title(FACTOR_NAMES[factor])

        sns.boxplot(data=d, x=factor, y=col, hue=factor, order=levels, hue_order=levels,
                    palette=colors, dodge=False, legend=False, width=0.68,
                    showfliers=False, ax=ax_box)
        sns.stripplot(data=d, x=factor, y=col, order=levels, color="#26323c",
                      size=2.2, jitter=0.22, alpha=0.26, ax=ax_box)
        if log_axis:
            ax_box.set_yscale("log")
        ax_box.set(title="Boxplots and participants" + ("  |  log y-axis" if log_axis else ""),
                   xlabel=FACTOR_NAMES[factor], ylabel=axis_label)

        sns.violinplot(data=d, x=factor, y=col, hue=factor, order=levels, hue_order=levels,
                       palette=colors, dodge=False, legend=False, inner="quartile", cut=0,
                       linewidth=1.0, ax=ax_violin)
        if log_axis:
            ax_violin.set_yscale("log")
        ax_violin.set(title="Violin distribution comparison" + ("  |  log y-axis" if log_axis else ""),
                      xlabel=FACTOR_NAMES[factor], ylabel=axis_label)

        for ax in (ax_box, ax_violin):
            ax.tick_params(axis="x", rotation=25 if factor == "bp_category" else 0)
        for ax in axs.flat:
            sns.despine(ax=ax)
        fig.suptitle(f"{label} across {FACTOR_NAMES[factor].lower()}", fontsize=19,
                     fontweight="semibold", y=0.985)
        fig.text(0.5, 0.943,
                 f"n = {len(d)}   |   Kruskal–Wallis q = {outcome['kruskal_q_bh_28_tests']:.3g}"
                 f"   |   ε² = {outcome['epsilon_squared']:.3f}",
                 ha="center", fontsize=11, color="#475569")
        fig.tight_layout(rect=(0, 0, 1, 0.927), h_pad=3.2, w_pad=2.2)
        fig.savefig(HRV_FIGURE_DIRS[col]["categorical"] / f"{factor}__distribution.png",
                    dpi=180, bbox_inches="tight")
        plt.close(fig)

print("Saved 28 detailed distribution figures organized by HRV variable.")

findfont: Failed to find font weight semibold for DejaVu Sans, now using 700.
findfont: Failed to find font weight semibold for DejaVu Sans, now using 700.


Saved 28 detailed distribution figures organized by HRV variable.


## 7. Factor overview figures and pairwise-effect matrices

The factor overview figure facilitates comparison of all seven HRV outcomes without combining the outcomes numerically. The Cliff's delta matrix reports signed pairwise effects in their original level ordering; it is not a correlation or a clustering result.

In [10]:
for factor, levels in FACTORS.items():
    colors = COLORS[factor]
    fig, axs = plt.subplots(2, 4, figsize=(22, 10.5))
    for ax, (col, (label, unit)) in zip(axs.flat[:7], HRV.items()):
        sns.boxplot(data=df, x=factor, y=col, hue=factor, order=levels, hue_order=levels,
                    palette=colors, dodge=False, legend=False, showfliers=False,
                    width=0.68, ax=ax)
        if use_log_axis(col):
            ax.set_yscale("log")
        ax.set(title=label + ("  (log scale)" if use_log_axis(col) else ""),
               xlabel="", ylabel=unit)
        ax.tick_params(axis="x", rotation=35 if factor == "bp_category" else 0,
                       labelsize=9)
        sns.despine(ax=ax)
    ax = axs.flat[7]
    n_by_level = [int((df[factor] == level).sum()) for level in levels]
    bars = ax.bar(levels, n_by_level, color=[colors[level] for level in levels], width=0.72)
    ax.bar_label(bars, fontsize=10, padding=3)
    ax.set(title="Group sample sizes", xlabel="", ylabel="Participants")
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.tick_params(axis="x", rotation=35 if factor == "bp_category" else 0,
                   labelsize=9)
    sns.despine(ax=ax)
    fig.suptitle(f"HRV overview by {FACTOR_NAMES[factor].lower()}",
                 fontweight="semibold", fontsize=21)
    fig.tight_layout(rect=(0, 0, 1, 0.95), h_pad=3, w_pad=2)
    fig.savefig(FACTOR_OVERVIEW_DIR / f"{factor}__overview.png", dpi=180, bbox_inches="tight")
    plt.close(fig)

    d = pairwise.loc[pairwise["factor"] == factor].copy()
    d["pair"] = d["level_1"] + " vs " + d["level_2"]
    matrix = d.pivot(index="hrv_variable", columns="pair", values="cliffs_delta_1_vs_2")
    matrix = matrix.reindex(index=list(HRV))
    fig_width = max(8, 1.35 * len(matrix.columns) + 4)
    fig, ax = plt.subplots(figsize=(fig_width, 6.7))
    sns.heatmap(matrix, cmap="vlag", center=0, vmin=-1, vmax=1,
                linewidths=0.45, linecolor="white", annot=True, fmt=".2f",
                cbar_kws={"label": "Cliff's delta (first group vs second)"},
                ax=ax)
    ax.set_yticklabels([HRV[c][0] for c in matrix.index], rotation=0)
    ax.set(xlabel="Group comparison", ylabel="", title=f"Pairwise Cliff's delta by {FACTOR_NAMES[factor].lower()}")
    ax.tick_params(axis="x", labelrotation=45, labelsize=9)
    for tick in ax.get_xticklabels():
        tick.set_ha("right")
    fig.tight_layout()
    fig.savefig(FACTOR_OVERVIEW_DIR / f"{factor}__cliffs_delta_heatmap.png",
                dpi=180, bbox_inches="tight")
    plt.close(fig)

print(f"Saved 4 factor overviews and 4 Cliff's delta heatmaps to {FACTOR_OVERVIEW_DIR}")

findfont: Failed to find font weight semibold for DejaVu Sans, now using 700.


Saved 4 factor overviews and 4 Cliff's delta heatmaps to ../../eda_results/06_unifactorial_hrv_analysis/figures/_global/factor_overviews


## 8. Continuous-factor figures

For each HRV-by-continuous-factor pair: **(left)** participant-level scatterplot colored according to its corresponding categorical factor, overlaid with a robust LOWESS smooth; **(right)** conditional medians and 25th–75th percentiles across empirical quantile bins. For heavily right-skewed positive HRV outcomes, the y-axis uses a logarithmic scale, with LOWESS fitted on log-values. The smooth is descriptive, not a fitted causal model. Binning is for visualization only and has no effect on the correlation tests.

In [11]:
def continuous_display(y):
    series = df[y].dropna()
    return bool((series.min() > 0) and (series.quantile(0.99) / series.quantile(0.01) >= 30))

for x, (x_label, x_unit, color_factor) in CONTINUOUS_FACTORS.items():
    palette = COLORS[color_factor]
    for y, (y_label, y_unit) in HRV.items():
        d = df[[x, y, color_factor]].dropna().copy()
        correlation = continuous.loc[(continuous["factor_continuous"] == x) &
                                     (continuous["hrv_variable"] == y)].iloc[0]
        log_y = continuous_display(y)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15.8, 6.0), gridspec_kw={"width_ratios": [1.22, 1]})

        sns.scatterplot(data=d, x=x, y=y, hue=color_factor,
                        hue_order=FACTORS[color_factor], palette=palette,
                        s=25, alpha=0.47, edgecolor="none", ax=ax1)
        x_values, y_values = d[x].to_numpy(), d[y].to_numpy()
        y_fit = np.log10(y_values) if log_y else y_values
        smooth = lowess(y_fit, x_values, frac=0.40, it=2, return_sorted=True)
        smooth_y = 10 ** smooth[:, 1] if log_y else smooth[:, 1]
        ax1.plot(smooth[:, 0], smooth_y, color="#172d42", lw=2.8, label="Robust LOWESS")
        ax1.set(title="Individual observations and smooth trend", xlabel=f"{x_label} ({x_unit})", ylabel=f"{y_label} ({y_unit})")
        handles, legend_labels = ax1.get_legend_handles_labels()
        if handles:
            ax1.legend(title=FACTOR_NAMES[color_factor], fontsize=8.5, title_fontsize=9,
                       loc="best", frameon=True, framealpha=0.91, ncol=1)
        if log_y:
            ax1.set_yscale("log")

        d["quantile_bin"] = pd.qcut(d[x], q=5, duplicates="drop")
        bins = (d.groupby("quantile_bin", observed=True)
                  .agg(x_mid=(x, "median"), y_median=(y, "median"),
                       y_low=(y, lambda z: z.quantile(0.25)),
                       y_high=(y, lambda z: z.quantile(0.75)), n=(y, "size"))
                  .reset_index(drop=True))
        ax2.errorbar(bins["x_mid"], bins["y_median"],
                     yerr=[bins["y_median"] - bins["y_low"], bins["y_high"] - bins["y_median"]],
                     fmt="o-", color="#214f6b", markerfacecolor="#45a4a9",
                     markersize=8.0, linewidth=2.0, capsize=5.0, elinewidth=1.7)
        for row in bins.itertuples():
            ax2.annotate(f"n={row.n}", (row.x_mid, row.y_median), xytext=(5, 8),
                         textcoords="offset points", fontsize=8.5, color="#334155")
        ax2.set(title="Conditional median and IQR by quantile", xlabel=f"{x_label} ({x_unit})", ylabel=f"{y_label} ({y_unit})")
        if log_y:
            ax2.set_yscale("log")
        for ax in (ax1, ax2):
            sns.despine(ax=ax)
        fig.suptitle(f"{y_label} vs {x_label}  |  n={len(d)}", fontsize=18,
                     fontweight="semibold", y=0.986)
        fig.text(0.5, 0.900,
                 f"Spearman ρ={correlation.spearman_rho:+.3f} "
                 f"[95% CI {correlation.spearman_ci95_lower:+.3f}, {correlation.spearman_ci95_upper:+.3f}]"
                 f"  |  FDR q={correlation.spearman_q_bh_28_tests:.3g}",
                 ha="center", fontsize=10.5, color="#475569")
        fig.tight_layout(rect=(0, 0, 1, 0.845), w_pad=2.2)
        fig.savefig(HRV_FIGURE_DIRS[y]["continuous"] / f"{x}.png", dpi=170, bbox_inches="tight")
        plt.close(fig)
print("Saved continuous-factor figures organized by HRV variable.")

findfont: Failed to find font weight semibold for DejaVu Sans, now using 700.


Saved continuous-factor figures organized by HRV variable.


## 9. Sensitivity analyses

Two **analysis-only** scenarios are considered: **(A)** omit factor levels with n < 10 (applies to blood-pressure Stage 3 only, n=4); **(B)** omit HRV observations below the 1st or above the 99th **overall HRV percentile** for each outcome. Each Kruskal test is rerun and BH-FDR-adjusted across the scenarios independently. These are exploratory robustness checks; their subsets are never written as cleaned datasets, and no participant is removed from the main analysis. Changes in p-values alone should not be interpreted as changes in biological effect.

In [12]:
sensitivity_rows = []
for factor, levels in FACTORS.items():
    observed_counts = df[factor].value_counts()
    supported_levels = [v for v in levels if observed_counts.get(v, 0) >= MIN_SENSITIVITY_GROUP_N]
    for y in HRV:
        d = df[[factor, y]].dropna()
        scenarios = []
        if len(supported_levels) != len(levels):
            scenarios.append(("remove_small_levels_n_lt_10", d.loc[d[factor].isin(supported_levels)]))
        q01, q99 = d[y].quantile([0.01, 0.99])
        scenarios.append(("trim_1pct_each_tail", d.loc[d[y].between(q01, q99, inclusive="both")]))
        primary = omnibus.loc[(omnibus["factor"] == factor) & (omnibus["hrv_variable"] == y)].iloc[0]
        for scenario, s in scenarios:
            groups = [s.loc[s[factor] == v, y].to_numpy() for v in levels if (s[factor] == v).any()]
            h, p = stats.kruskal(*groups)
            n = len(s)
            k = len(groups)
            epsilon = max(0.0, (h - k + 1) / (n - k))
            sensitivity_rows.append({
                "factor": factor, "hrv_variable": y, "scenario": scenario,
                "n_primary": len(d), "n_sensitivity": n, "n_omitted": len(d) - n,
                "n_groups": k, "minimum_group_n": min(map(len, groups)),
                "kruskal_h": h, "kruskal_p": p, "epsilon_squared": epsilon,
                "primary_kruskal_p": primary["kruskal_p"],
                "primary_epsilon_squared": primary["epsilon_squared"],
                "epsilon_squared_change": epsilon - primary["epsilon_squared"],
            })
sensitivity = pd.DataFrame(sensitivity_rows)
sensitivity["kruskal_q_bh_within_scenario"] = np.nan
for _, block in sensitivity.groupby("scenario", sort=False):
    sensitivity.loc[block.index, "kruskal_q_bh_within_scenario"] = multipletests(block["kruskal_p"], method="fdr_bh")[1]
sensitivity["primary_kruskal_q_bh_28_tests"] = [omnibus.loc[(omnibus["factor"] == row.factor) & (omnibus["hrv_variable"] == row.hrv_variable), "kruskal_q_bh_28_tests"].iloc[0] for row in sensitivity.itertuples()]
sensitivity.to_csv(OUTPUT_DIR / "sensitivity_checks.csv", index=False)
print(f"Sensitivity checks: {len(sensitivity)}")
display(sensitivity.head(14).round(4))

Sensitivity checks: 35


,factor,hrv_variable,scenario,n_primary,n_sensitivity,n_omitted,n_groups,minimum_group_n,kruskal_h,kruskal_p,epsilon_squared,primary_kruskal_p,primary_epsilon_squared,epsilon_squared_change,kruskal_q_bh_within_scenario,primary_kruskal_q_bh_28_tests
0,age_group,t2m_pre_mean_rr,trim_1pct_each_tail,476,466,10,3,37,9.5749,0.0083,0.0164,0.0171,0.0130,0.0034,0.0353,0.0598
1,age_group,t2m_pre_mean_hr,trim_1pct_each_tail,476,468,8,3,37,9.4629,0.0088,0.0160,0.0156,0.0134,0.0027,0.0353,0.0598
2,age_group,t2m_pre_sdnn,trim_1pct_each_tail,476,466,10,3,35,4.6047,0.1000,0.0056,0.0387,0.0095,-0.0039,0.1750,0.0984
3,age_group,t2m_pre_rmssd,trim_1pct_each_tail,476,466,10,3,36,6.0712,0.0480,0.0088,0.1356,0.0042,0.0046,0.0961,0.2420
4,age_group,t2m_pre_hf,trim_1pct_each_tail,476,470,6,3,37,12.0439,0.0024,0.0215,0.0022,0.0217,-0.0002,0.0189,0.0151
5,age_group,t2m_pre_lf,trim_1pct_each_tail,476,466,10,3,34,8.9969,0.0111,0.0151,0.0011,0.0247,-0.0096,0.0389,0.0099
6,age_group,t2m_pre_vlf,trim_1pct_each_tail,476,470,6,3,37,7.9515,0.0188,0.0127,0.0293,0.0107,0.0020,0.0438,0.0820
7,sex_category,t2m_pre_mean_rr,trim_1pct_each_tail,476,466,10,2,139,0.0466,0.8291,0.0000,0.8342,0.0000,0.0000,0.8929,0.8984
8,sex_category,t2m_pre_mean_hr,trim_1pct_each_tail,476,468,8,2,140,0.1147,0.7348,0.0000,0.7935,0.0000,0.0000,0.8230,0.8887
9,sex_category,t2m_pre_sdnn,trim_1pct_each_tail,476,466,10,2,137,45.6217,0.0000,0.0962,0.0000,0.0871,0.0091,0.0000,0.0000


## 10. Cross-factor synthesis

An HRV-by-factor table combines **global categorical effects**, uncertainty, adjusted p-values, and the largest eligible pairwise Cliff effect among pairs with **both groups n ≥ 10**. A second table/matrix displays continuous Spearman associations. The global effect magnitude is unsigned, while pairwise Cliff's delta and continuous Spearman are signed. Heatmap asterisks indicate FDR q < 0.05 but do not substitute for confidence intervals or indicate clinical importance.

In [13]:
summary_rows = []
for row in omnibus.itertuples():
    eligible = pairwise.loc[(pairwise["factor"] == row.factor) &
                            (pairwise["hrv_variable"] == row.hrv_variable) &
                            (pairwise["n_1"] >= MIN_SENSITIVITY_GROUP_N) &
                            (pairwise["n_2"] >= MIN_SENSITIVITY_GROUP_N)]
    if len(eligible):
        best = eligible.loc[eligible["cliffs_delta_1_vs_2"].abs().idxmax()]
        pair_name = f"{best.level_1} vs {best.level_2}"
        pair_delta = best.cliffs_delta_1_vs_2
        pair_low, pair_high = best.cliffs_delta_ci95_lower, best.cliffs_delta_ci95_upper
        pair_q = best.comparison_q_bh_all_175_tests
    else:
        pair_name, pair_delta, pair_low, pair_high, pair_q = "", np.nan, np.nan, np.nan, np.nan
    ordered = trends.loc[(trends["factor"] == row.factor) & (trends["hrv_variable"] == row.hrv_variable)]
    ord_rho = ordered["spearman_rho"].iloc[0] if len(ordered) else np.nan
    ord_q = ordered["q_bh_21_tests"].iloc[0] if len(ordered) else np.nan
    summary_rows.append({
        "hrv_variable": row.hrv_variable, "factor": row.factor, "n": row.n,
        "smallest_category_n": row.smallest_group_n, "small_category_caution": row.small_group_caution,
        "kruskal_epsilon_squared": row.epsilon_squared,
        "epsilon_squared_ci95_lower": row.epsilon_squared_ci95_lower,
        "epsilon_squared_ci95_upper": row.epsilon_squared_ci95_upper,
        "kruskal_q_bh_28_tests": row.kruskal_q_bh_28_tests,
        "brown_forsythe_q_bh_28_tests": row.brown_forsythe_q_bh_28_tests,
        "strongest_eligible_pair": pair_name,
        "strongest_eligible_pair_cliffs_delta": pair_delta,
        "strongest_eligible_pair_cliffs_ci95_lower": pair_low,
        "strongest_eligible_pair_cliffs_ci95_upper": pair_high,
        "strongest_eligible_pair_q_bh_all_175": pair_q,
        "ordered_factor_rho": ord_rho, "ordered_factor_q_bh_21": ord_q,
    })
association_summary = pd.DataFrame(summary_rows)
association_summary.to_csv(OUTPUT_DIR / "hrv_factor_association_summary.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(18.3, 6.5), gridspec_kw={"width_ratios": [1, 1.15]})
left = omnibus.pivot(index="hrv_variable", columns="factor", values="epsilon_squared").reindex(index=list(HRV), columns=list(FACTORS))
left_q = omnibus.pivot(index="hrv_variable", columns="factor", values="kruskal_q_bh_28_tests").reindex(index=list(HRV), columns=list(FACTORS))
annot_left = np.array([[f"{left.iloc[i, j]:.2f}" + ("*" if left_q.iloc[i, j] < 0.05 else "")
                       for j in range(len(left.columns))] for i in range(len(left.index))])
sns.heatmap(left, ax=axes[0], cmap="YlGnBu", vmin=0, linewidths=0.5, annot=annot_left, fmt="",
            cbar_kws={"label": "Kruskal epsilon-squared"})
axes[0].set(title="Categorical factors: global effect magnitude", xlabel="Factor", ylabel="")
axes[0].set_xticklabels([FACTOR_NAMES[f] for f in left.columns], rotation=30, ha="right")
axes[0].set_yticklabels([HRV[h][0] for h in left.index], rotation=0)

right = continuous.pivot(index="hrv_variable", columns="factor_continuous", values="spearman_rho").reindex(index=list(HRV), columns=list(CONTINUOUS_FACTORS))
right_q = continuous.pivot(index="hrv_variable", columns="factor_continuous", values="spearman_q_bh_28_tests").reindex(index=list(HRV), columns=list(CONTINUOUS_FACTORS))
annot_right = np.array([[f"{right.iloc[i, j]:+.2f}" + ("*" if right_q.iloc[i, j] < 0.05 else "")
                        for j in range(len(right.columns))] for i in range(len(right.index))])
sns.heatmap(right, ax=axes[1], cmap="vlag", center=0, vmin=-0.6, vmax=0.6,
            linewidths=0.5, annot=annot_right, fmt="", cbar_kws={"label": "Spearman rho"})
axes[1].set(title="Continuous factors: monotonic associations", xlabel="Factor", ylabel="")
axes[1].set_xticklabels([CONTINUOUS_FACTORS[f][0] for f in right.columns], rotation=30, ha="right")
axes[1].set_yticklabels([HRV[h][0] for h in right.index], rotation=0)
fig.suptitle("HRV associations with individual factors", fontsize=19, fontweight="semibold", y=1.035)
fig.text(0.5, -0.015, "* FDR q < 0.05 within each testing family; association does not imply causation.",
         ha="center", fontsize=10, color="#475569")
fig.tight_layout()
fig.savefig(SYNTHESIS_FIGURE_DIR / "hrv_factor_association_synthesis.png", dpi=180, bbox_inches="tight")
plt.close(fig)

fig, ax = plt.subplots(figsize=(10.4, 6.3))
sensitivity_delta = sensitivity.pivot_table(index="hrv_variable", columns=["scenario", "factor"], values="epsilon_squared_change")
# Summarize only sensitivity effects that change the tested sample.
changed = sensitivity.loc[sensitivity["n_omitted"] > 0].copy()
changed["label"] = changed["factor"].map(FACTOR_NAMES) + " · " + changed["scenario"].map({
    "remove_small_levels_n_lt_10": "omit n<10 level", "trim_1pct_each_tail": "1–99% trim"})
delta_matrix = changed.pivot(index="hrv_variable", columns="label", values="epsilon_squared_change").reindex(index=list(HRV))
# Trimming may generate many columns; display the actual observed scenarios, not a fabricated comparison.
sns.heatmap(delta_matrix, center=0, cmap="vlag", annot=True, fmt="+.3f", linewidths=0.4,
            cbar_kws={"label": "Change in epsilon-squared vs primary"}, ax=ax)
ax.set(title="Sensitivity of rank-based global effects", xlabel="Analysis-only sensitivity scenario", ylabel="")
ax.set_yticklabels([HRV[h][0] for h in delta_matrix.index], rotation=0)
ax.tick_params(axis="x", rotation=45, labelsize=8)
for t in ax.get_xticklabels():
    t.set_ha("right")
fig.tight_layout()
fig.savefig(SYNTHESIS_FIGURE_DIR / "sensitivity_effect_change_heatmap.png", dpi=180, bbox_inches="tight")
plt.close(fig)
print("Saved association summary table and 2 synthesis heatmaps.")
display(association_summary.head(12).round(4))

Saved association summary table and 2 synthesis heatmaps.


,hrv_variable,factor,n,smallest_category_n,small_category_caution,kruskal_epsilon_squared,epsilon_squared_ci95_lower,epsilon_squared_ci95_upper,kruskal_q_bh_28_tests,brown_forsythe_q_bh_28_tests,strongest_eligible_pair,strongest_eligible_pair_cliffs_delta,strongest_eligible_pair_cliffs_ci95_lower,strongest_eligible_pair_cliffs_ci95_upper,strongest_eligible_pair_q_bh_all_175,ordered_factor_rho,ordered_factor_q_bh_21
0,t2m_pre_mean_rr,age_group,476,37,False,0.0130,0.0000,0.0436,0.0598,0.5800,60-69 vs 80+,-0.2058,-0.3990,0.0026,0.3137,0.1303,0.0232
1,t2m_pre_mean_hr,age_group,476,37,False,0.0134,0.0000,0.0429,0.0598,0.7189,60-69 vs 80+,0.2058,-0.0045,0.3948,0.3137,-0.1316,0.0232
2,t2m_pre_sdnn,age_group,476,37,False,0.0095,0.0000,0.0381,0.0984,0.5800,60-69 vs 80+,0.2539,0.0492,0.4573,0.1726,-0.1034,0.0844
3,t2m_pre_rmssd,age_group,476,37,False,0.0042,0.0000,0.0281,0.2420,0.7411,60-69 vs 80+,0.2083,0.0074,0.4046,0.3137,-0.0594,0.4565
4,t2m_pre_hf,age_group,476,37,False,0.0217,0.0016,0.0594,0.0151,0.7259,60-69 vs 80+,0.3314,0.1247,0.5089,0.0633,-0.1556,0.0069
5,t2m_pre_lf,age_group,476,37,False,0.0247,0.0046,0.0628,0.0099,0.7259,60-69 vs 80+,0.3396,0.1483,0.5377,0.0348,-0.1599,0.0069
6,t2m_pre_vlf,age_group,476,37,False,0.0107,0.0000,0.0401,0.0820,0.5800,60-69 vs 80+,0.2613,0.0683,0.4515,0.1726,-0.1131,0.0571
7,t2m_pre_mean_rr,sex_category,476,143,False,0.0000,0.0000,0.0123,0.8984,0.5800,Female vs Male,-0.0121,-0.1289,0.0997,0.9301,NaN,NaN
8,t2m_pre_mean_hr,sex_category,476,143,False,0.0000,0.0000,0.0102,0.8887,0.5800,Female vs Male,0.0151,-0.1005,0.1333,0.9079,NaN,NaN
9,t2m_pre_sdnn,sex_category,476,143,False,0.0871,0.0404,0.1527,0.0000,0.5800,Female vs Male,0.3757,0.2677,0.4904,0.0000,NaN,NaN


## 11. Output validation and methodological cautions

- The primary analyses always use the complete Notebook 03 cohort. Sensitivity subsets are in-memory only and are never exported as modified datasets.
- **Kruskal–Wallis** is a global rank-based distribution test; differences should not automatically be interpreted as differences in means/medians when group distributions differ in shape.
- **Dunn** adjusts pooled ranks for ties. Post hoc p-values are FDR-adjusted both within each HRV–factor family and across all 175 exploratory tests, and the two scopes are labeled separately.
- **Cliff's delta** is signed for the order `level_1` relative to `level_2`; effect size bootstraps resample the groups separately. For groups with n < 10, percentile intervals are exploratory and potentially unstable.
- **Global epsilon-squared** is clipped at zero and bootstrapped within groups. The bootstrap interval describes uncertainty of this exploratory rank-based effect; very small groups may distort it.
- **Spearman** on categorical ordered factors assigns ordinal scores and is distinct from Spearman on original continuous measurements. Age, BMI, SBP and DBP are not jointly adjusted here.
- **Lowess lines and quantile bins** are visualization tools only; they are not inferential models.
- **Small n:** blood-pressure Stage 3 has 4 observations. It remains visible in the primary comparisons. The small-group sensitivity omits this level temporarily without changing dataset composition.
- **Tail sensitivity:** values outside the global 1st–99th percentiles are temporarily omitted per HRV outcome for robustness checks; no physiological values are declared invalid.
- **Multiplicity:** 28 global categorical KW tests, 28 Brown–Forsythe tests, 21 ordered-factor trends, 28 continuous Spearman tests, 175 pairwise location and 175 KS comparisons have distinct labeled FDR scopes.
- Physiological metrics, especially Mean RR and Mean HR, are not statistically independent. These tests do not assess HRV-to-HRV relationships or combined predictors.

In [14]:
expected_descriptives = len(HRV) * sum(len(levels) for levels in FACTORS.values())
expected_pairwise = len(HRV) * sum(len(levels) * (len(levels) - 1) // 2 for levels in FACTORS.values())
assert len(group_summary) == expected_descriptives
assert len(omnibus) == len(HRV) * len(FACTORS)
assert len(pairwise) == expected_pairwise
assert len(trends) == len(HRV) * 3
assert len(continuous) == len(HRV) * len(CONTINUOUS_FACTORS)
assert len(association_summary) == len(HRV) * len(FACTORS)
assert int((continuous["n"] > 0).sum()) == 28
assert len(sensitivity) == 35
assert len(list(FACTOR_OVERVIEW_DIR.glob("*.png"))) == 8
assert len(list(SYNTHESIS_FIGURE_DIR.glob("*.png"))) == 2
assert len(list(FIGURE_DIR.rglob("categorical/*.png"))) == 28
assert len(list(FIGURE_DIR.rglob("continuous/*.png"))) == 28
assert len(df) == 476, "The cohort size changed unexpectedly."
for d in (omnibus, pairwise, continuous):
    assert not d.empty

csv_outputs = sorted(OUTPUT_DIR.glob("*.csv"))
print("Completed — 06 Unifactorial HRV analysis | Extended EDA")
print(f"Participants (unchanged): {len(df)}")
print(f"Categorical tests: {len(omnibus)}; pairwise contrasts: {len(pairwise)}")
print(f"Continuous tests: {len(continuous)}; sensitivity checks: {len(sensitivity)}")
print(f"CSV outputs: {len(csv_outputs)}")
print(f"PNG figures: {len(list(FIGURE_DIR.rglob('*.png')))}")
print(f"Per-HRV figure folders: {[HRV_FIGURE_DIRS[c]['base'].name for c in HRV]}")
print(f"Global figure folders: {FACTOR_OVERVIEW_DIR} and {SYNTHESIS_FIGURE_DIR}")
print(f"Results folder: {OUTPUT_DIR}")

Completed — 06 Unifactorial HRV analysis | Extended EDA
Participants (unchanged): 476
Categorical tests: 28; pairwise contrasts: 175
Continuous tests: 28; sensitivity checks: 35
CSV outputs: 8
PNG figures: 66
Per-HRV figure folders: ['mean_rr', 'mean_hr', 'sdnn', 'rmssd', 'hf', 'lf', 'vlf']
Global figure folders: ../../eda_results/06_unifactorial_hrv_analysis/figures/_global/factor_overviews and ../../eda_results/06_unifactorial_hrv_analysis/figures/_global/synthesis
Results folder: ../../eda_results/06_unifactorial_hrv_analysis
